# 00 · Data Ingestion — Refinitiv Workspace / Eikon

**Run this notebook top to bottom before any analysis notebook.**

| Step | What | Est. API calls |
|---|---|---|
| 1 | Load 2,182 pre-reform TOPIX constituents from JPX CSV | 0 |
| 2 | Fundamental snapshot for pre-reform stocks (2021) | ~22 |
| 3 | Daily CLOSE + VOLUME — primary window (Apr 2021 – Mar 2022) | ~220 |
| 4 | Build original TOPIX with weights | 0 |
| 5 | Load 1,674 current TOPIX constituents from JPX CSV | 0 |
| 6 | TSE-wide FF market cap snapshot (identify inclusion candidates) | ~90 |
| 7 | Daily CLOSE + VOLUME — robustness window (Apr 2025 – Mar 2026) | ~250 |
| 8 | Apply continuation + inclusion criteria → build pro-forma | 0 |
| 9 | Meta (sector, name) for pro-forma stocks | ~15 |
| 10 | Daily BID + ASK — robustness window for pro-forma stocks | ~120 |

All outputs go to `./data/` (created automatically). Checkpoint files prevent re-fetching.

In [ ]:
import eikon as ek
import pandas as pd
import numpy as np
import math
import os, time, warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)

# ── App key & data folder ─────────────────────────────────────────────────────
APP_KEY     = 'YOUR_APP_KEY_HERE'          # <── paste your key here (keep the quotes)
PROJECT_DIR = r'C:\Users\moolb\OneDrive\Dokumente\HSG\Bachelorarbeit\Python Analyses'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

ek.set_app_key(APP_KEY)

# quick connectivity check
try:
    _test, _ = ek.get_data('AAPL.O', ['TR.PriceClose'])
    print('Eikon connected successfully.')
except Exception as e:
    print(f'Connection error: {e}')
    print('Make sure Refinitiv Workspace is open and running in the background.')

In [ ]:
# ── Time windows ──────────────────────────────────────────────────────────────
START_P = '2021-04-01'   # primary window start (historical — kept for context only)
END_P   = '2022-03-31'   # primary window end
START_R = '2025-04-01'   # robustness window start (primary analysis window)
END_R   = '2026-03-31'   # robustness window end

BATCH = 10               # Eikon timeseries limit per call (keep low to avoid truncation)
META_BATCH = 100         # Eikon get_data limit per call
SLEEP = 0.6              # seconds between batches
FULL_YEAR_DAYS = 245     # TSE trading days per year (for ATVR annualisation)
MIN_COVERAGE   = 0.85    # minimum data coverage for ATVR

def p(path):
    """Return full path inside DATA_DIR. Auto-creates the folder if missing."""
    os.makedirs(DATA_DIR, exist_ok=True)
    return os.path.join(DATA_DIR, path)

def already_saved(name):
    """Return True if checkpoint file exists — skip re-pulling."""
    exists = os.path.exists(p(name))
    if exists:
        print(f'  [SKIP] {name} already saved — delete file to re-pull.')
    return exists

# ── Robust Eikon column renaming ──────────────────────────────────────────────
def rename_eikon_cols(df, friendly_names):
    """
    Rename columns returned by ek.get_data() to friendly names.
    First column ('Instrument') → 'RIC', then positional mapping.
    Handles extra/missing columns gracefully instead of crashing.
    """
    expected = ['RIC'] + list(friendly_names)
    actual_n = len(df.columns)
    expected_n = len(expected)

    if actual_n == expected_n:
        df.columns = expected
    elif actual_n > expected_n:
        print(f'    WARNING: Eikon returned {actual_n} cols, expected {expected_n}. Dropping {actual_n - expected_n} extra col(s).')
        rename_map = {df.columns[i]: expected[i] for i in range(expected_n)}
        df = df.rename(columns=rename_map)
        extra_cols = [c for c in df.columns if c not in expected]
        df = df.drop(columns=extra_cols)
    else:
        print(f'    WARNING: Eikon returned {actual_n} cols, expected {expected_n}. Adding {expected_n - actual_n} missing col(s) as NaN.')
        df.columns = expected[:actual_n]
        for col in expected[actual_n:]:
            df[col] = np.nan

    return df

# ── Eikon time-series fetcher ─────────────────────────────────────────────────
def fetch_ts(rics_list, fields, start, end, label=''):
    """
    Fetch Eikon time-series in batches of BATCH RICs.
    Returns a dict {field: wide_DataFrame}.
    """
    frames  = {f: [] for f in fields}
    n       = len(rics_list)
    n_batch = (n + BATCH - 1) // BATCH
    errors  = []

    for i, start_i in enumerate(range(0, n, BATCH)):
        batch = rics_list[start_i : start_i + BATCH]
        try:
            ts = ek.get_timeseries(
                batch, fields=fields,
                start_date=start, end_date=end,
                interval='daily'
            )
            if ts is None or ts.empty:
                continue
            # MultiIndex when >1 RIC, flat columns when exactly 1 RIC
            if isinstance(ts.columns, pd.MultiIndex):
                for f in fields:
                    if f in ts.columns.get_level_values(1):
                        frames[f].append(ts.xs(f, axis=1, level=1))
            else:
                for f in fields:
                    if f in ts.columns:
                        frames[f].append(
                            ts[[f]].rename(columns={f: batch[0]})
                        )
        except Exception as e:
            errors.append((i, str(e)))

        if (i + 1) % 20 == 0 or (i + 1) == n_batch:
            print(f'  {label}  {i+1}/{n_batch} batches  '
                  f'({(i+1)/n_batch*100:.0f}%)  errors so far: {len(errors)}')
        time.sleep(SLEEP)

    if errors:
        print(f'  {len(errors)} failed batches: {errors[:5]}')

    return {f: pd.concat(frames[f], axis=1) if frames[f] else pd.DataFrame()
            for f in fields}

# ──────────────────────────────────────────────────────────────────────────────
# FREE-FLOAT WEIGHT METHODOLOGY
# ──────────────────────────────────────────────────────────────────────────────
# Two weighting methodologies are implemented. Which one is applied is controlled
# by the `which` argument of `apply_ffw()` — there is no silent default.
#
#   compute_postreform_ffw — Phase-2 reform "Adjusted Free-Float Weight"
#     Adjusted Float % = (SharesOut − Treasury − Strategic) / SharesOut,
#     rounded UP to the nearest 0.05.
#     Falls back to FreeFloatPct / 100 (also rounded up) when strategic-holder
#     data is missing. This is the methodology TSE introduced with the 2022
#     reform and uses for the current TOPIX and for the pro-forma index.
#
#   compute_prereform_ffw — pre-reform TOPIX weighting proxy
#     FFW = FreeFloatPct / 100. No explicit strategic-holder deduction and no
#     5-percentage-point rounding. Refinitiv TR.FreeFloatPct is used as a
#     practical proxy for the pre-2022 TSE FFW bucket system, which is not
#     exposed via Eikon. This is the methodology applied to the pre-reform
#     membership when building the "Original-Today" counterfactual index.
#
# Both functions return (ffw_value, fallback_flag). apply_ffw() dispatches and
# writes the result to the columns `Adjusted_FFW` / `ffm_fallback` — those
# column names are preserved for backward compatibility with existing parquet
# files and analysis notebooks. The methodology applied is recorded by WHICH
# function was called, not by the column name.
#
# The back-compat names `compute_adjusted_ffw` / `apply_adjusted_ffw` are
# retained as thin wrappers that dispatch to the post-reform path. Earlier
# cells (Steps 2, 4, 6, 8, 9) still call them and therefore continue to work
# without modification.
# ──────────────────────────────────────────────────────────────────────────────

FFW_FIELDS = [
    'TR.SharesOutstanding',
    'TR.TreasurySharesOutstanding',     # treasury shares (Eikon field name)
    'TR.PctSharesHeldbyStratHolders',   # strategic holders % (closest Eikon equivalent)
]


def compute_postreform_ffw(row):
    """
    Post-reform (Phase-2) TOPIX Adjusted Free-Float Weight.
    Adjusted Float % = (SharesOut − Treasury − Strategic) / SharesOut,
    rounded UP to the nearest 0.05. Falls back to FreeFloatPct / 100
    (rounded up to 0.05) if strategic or SharesOut data is missing.
    Returns (ffw, fallback_flag).
    """
    shares_out = row.get('SharesOutstanding')
    treasury   = row.get('TreasuryShares')
    strat_pct  = row.get('StrategicHoldersPct')

    if pd.isna(shares_out) or shares_out <= 0:
        ff_pct = row.get('FreeFloatPct')
        if pd.notna(ff_pct) and ff_pct > 0:
            raw = ff_pct / 100.0
            return math.ceil(raw / 0.05) * 0.05, True
        return np.nan, True

    treasury = treasury if pd.notna(treasury) else 0.0

    if pd.notna(strat_pct) and strat_pct >= 0:
        strategic_shares = shares_out * strat_pct / 100.0
        raw = (shares_out - treasury - strategic_shares) / shares_out
    else:
        ff_pct = row.get('FreeFloatPct')
        if pd.notna(ff_pct) and ff_pct > 0:
            raw = ff_pct / 100.0
            return math.ceil(raw / 0.05) * 0.05, True
        return np.nan, True

    raw = max(raw, 0.0)
    raw = min(raw, 1.0)
    return math.ceil(raw / 0.05) * 0.05, False


def compute_prereform_ffw(row):
    """
    Pre-reform TOPIX weighting proxy.
    FFW = FreeFloatPct / 100. No strategic-holder deduction. No rounding.
    (Refinitiv TR.FreeFloatPct is used as proxy for the pre-2022 TSE FFW
    bucket system, which is not exposed via Eikon.)
    Returns (ffw, fallback_flag). fallback_flag=True iff FreeFloatPct is
    NaN or non-positive, in which case ffw=NaN.
    """
    ff_pct = row.get('FreeFloatPct')
    if pd.notna(ff_pct) and ff_pct > 0:
        return ff_pct / 100.0, False
    return np.nan, True


def apply_ffw(df, which='postreform'):
    """
    Apply a free-float weighting function to a DataFrame that has columns:
      SharesOutstanding, TreasuryShares, StrategicHoldersPct, FreeFloatPct
    Adds columns: Adjusted_FFW, ffm_fallback
    `which` ∈ {'prereform', 'postreform'}.
    Column names are kept as 'Adjusted_FFW' / 'ffm_fallback' for backward
    compatibility with existing parquet files and analysis notebooks.
    """
    if which == 'postreform':
        fn = compute_postreform_ffw
    elif which == 'prereform':
        fn = compute_prereform_ffw
    else:
        raise ValueError(f"which must be 'prereform' or 'postreform', got {which!r}")
    results = df.apply(fn, axis=1, result_type='expand')
    df['Adjusted_FFW']  = results[0].astype(float)
    df['ffm_fallback']  = results[1].astype(bool)
    return df


# ── Back-compat aliases ───────────────────────────────────────────────────────
# Earlier cells (Steps 2, 4, 6, 8, 9) call these names. They dispatch to the
# post-reform path, matching the original behaviour of the code.
compute_adjusted_ffw = compute_postreform_ffw

def apply_adjusted_ffw(df):
    """Back-compat alias of apply_ffw(df, which='postreform')."""
    return apply_ffw(df, which='postreform')


def compute_ff_mktcap(df, price_col='AvgClose'):
    """
    FF_MktCap = SharesOutstanding × Adjusted_FFW × reference-month avg close.
    Falls back to CompanyMarketCap × Adjusted_FFW for stocks missing avg close.
    Works identically for both pre-reform and post-reform FFW methodologies —
    the methodology is embedded in the Adjusted_FFW column, not here.
    """
    df['FF_MktCap'] = df['SharesOutstanding'] * df['Adjusted_FFW'] * df[price_col]
    mask = df['FF_MktCap'].isna() & df['CompanyMarketCap'].notna() & df['Adjusted_FFW'].notna()
    df.loc[mask, 'FF_MktCap'] = df.loc[mask, 'CompanyMarketCap'] * df.loc[mask, 'Adjusted_FFW']
    return df


def round_up_005(x):
    """Round UP to nearest 0.05."""
    if pd.isna(x):
        return np.nan
    return math.ceil(x / 0.05) * 0.05


def ref_month_range(end_date_str):
    """
    Return (start, end) of the last full calendar month before end_date_str.
    E.g. if END_P = '2022-03-31' → ('2022-03-01', '2022-03-31').
    If END_R = '2026-03-31' → ('2026-03-01', '2026-03-31').
    """
    end_dt = pd.Timestamp(end_date_str)
    if end_dt == end_dt + pd.offsets.MonthEnd(0):
        m_start = end_dt.replace(day=1)
        m_end   = end_dt
    else:
        m_end   = end_dt.replace(day=1) - pd.Timedelta(days=1)
        m_start = m_end.replace(day=1)
    return m_start.strftime('%Y-%m-%d'), m_end.strftime('%Y-%m-%d')


def read_parquet_series(path, name='value'):
    """Read a parquet file and return a named Series, handling scalar/DataFrame edge cases."""
    df = pd.read_parquet(path)
    if df.shape[1] == 1:
        s = df.iloc[:, 0]
    else:
        s = df.stack().droplevel(0)
    if not isinstance(s, pd.Series):
        s = pd.Series([s]) if not hasattr(s, '__len__') else pd.Series(s)
    s.name = name
    return s


# ──────────────────────────────────────────────────────────────────────────────
# ATVR AND DENOMINATOR HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def two_snapshot_denom(ff_start, ff_end):
    """
    ATVR denominator: average of FF market cap at window start and window end.
    Where only one snapshot is available, use that one. Where both are NaN or
    non-positive, result is NaN.
      ff_start, ff_end : pandas Series indexed by RIC
    Returns: Series named 'FF_MktCap_denom' indexed by union of both inputs.
    """
    all_rics = sorted(set(ff_start.index.tolist()) | set(ff_end.index.tolist()))
    s = ff_start.reindex(all_rics)
    e = ff_end.reindex(all_rics)
    denom = pd.Series(index=all_rics, dtype=float, name='FF_MktCap_denom')
    both = s.notna() & e.notna()
    denom[both] = (s[both] + e[both]) / 2
    only_s = s.notna() & e.isna()
    only_e = s.isna() & e.notna()
    denom[only_s] = s[only_s]
    denom[only_e] = e[only_e]
    denom[denom <= 0] = np.nan
    return denom


def compute_atvr(close_df, volume_df, denom,
                 min_cov=MIN_COVERAGE, trading_days=FULL_YEAR_DAYS):
    """
    Per-stock ATVR with coverage filtering.
      close_df, volume_df : wide DataFrames (index=dates, cols=RICs)
      denom               : Series of FF market-cap denominator indexed by RIC
      min_cov             : stocks with coverage < min_cov receive ATVR=NaN
      trading_days        : assumed trading days per year for annualisation
    Returns: (atvr_series, coverage_df).
      atvr_series indexed by RIC. NaN for stocks below min_cov.
      coverage_df has columns RIC / valid_days / coverage.
    """
    # Align RIC columns between close and volume
    rics_common = close_df.columns.intersection(volume_df.columns)
    close_df  = close_df[rics_common]
    volume_df = volume_df[rics_common]

    total_days = close_df.shape[0]
    valid_mask = close_df.notna() & volume_df.notna()
    valid_days = valid_mask.sum(axis=0)
    if total_days > 0:
        coverage = valid_days / total_days
    else:
        coverage = pd.Series(0.0, index=valid_days.index)

    daily_tv = (close_df * volume_df).where(valid_mask)
    sum_tv   = daily_tv.sum(axis=0)

    scalar = trading_days / valid_days.replace(0, np.nan)
    ann_tv = sum_tv * scalar

    denom_aligned = denom.reindex(ann_tv.index)
    atvr = (ann_tv / denom_aligned).astype(float)
    atvr[coverage < min_cov] = np.nan
    atvr.name = 'ATVR'

    cov_df = pd.DataFrame({
        'RIC': valid_days.index,
        'valid_days': valid_days.values,
        'coverage': coverage.values,
    })
    return atvr, cov_df


def load_robust_all(kind='prices'):
    """
    Load robust-window time series (CLOSE or VOLUME), concatenating the main
    pull (prices_robust / volume_robust) with the gap pull
    (prices_robust_gap / volume_robust_gap). The gap pull covers pre-reform
    RICs that are not part of the current-TOPIX + Stage-2-candidates main
    fetch, and is required for the Original-Today counterfactual index.
    If the gap file does not exist, only the main file is returned.
    Duplicate columns (RICs present in both files) are de-duplicated keeping
    the first occurrence.
      kind ∈ {'prices', 'volume'}
    """
    if kind == 'prices':
        main_file, gap_file = 'prices_robust.parquet', 'prices_robust_gap.parquet'
    elif kind == 'volume':
        main_file, gap_file = 'volume_robust.parquet', 'volume_robust_gap.parquet'
    else:
        raise ValueError(f"kind must be 'prices' or 'volume', got {kind!r}")
    frames = []
    if os.path.exists(p(main_file)):
        frames.append(pd.read_parquet(p(main_file)))
    if os.path.exists(p(gap_file)):
        frames.append(pd.read_parquet(p(gap_file)))
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, axis=1)
    df = df.loc[:, ~df.columns.duplicated()]
    return df


print(f'Primary window   : {START_P} → {END_P}    (historical; kept for context)')
print(f'Robustness window: {START_R} → {END_R}    (primary analysis window)')
print(f'Reference month (primary)   : {ref_month_range(END_P)}')
print(f'Reference month (robustness): {ref_month_range(END_R)}')
print(f'Min coverage for ATVR: {MIN_COVERAGE:.0%}')


In [ ]:
# ── Fresh-start cleanup (OFF by default) ──────────────────────────────────────
# This cell exists so a reviewer can force a full rebuild of derived files.
# DRY_RUN=True (default) → print what *would* be deleted but touch nothing.
# DRY_RUN=False          → actually remove the files, next run rebuilds them.
#
# Safe to leave DRY_RUN=True: the notebook is idempotent thanks to already_saved(),
# so nothing gets re-fetched unnecessarily.
DRY_RUN = True

OLD_FILES = [
    # Robustness data (may be truncated from old BATCH=40)
    'prices_robust.parquet',
    'volume_robust.parquet',
    'bid_robust.parquet',
    'ask_robust.parquet',
    # Robustness gap files for pre-reform RICs (Step 11)
    'prices_robust_gap.parquet',
    'volume_robust_gap.parquet',
    # Old pro-forma (was built from pre-reform list, now rebuilt from current TOPIX)
    'constituents_proforma.parquet',
    # Old original (primary-window; kept for context, NOT the counterfactual)
    'constituents_original.parquet',
    # Original-today counterfactual (rebuilt from new Steps 13-14)
    'constituents_original_today.parquet',
    'meta_original_today.parquet',
    'meta_original_endr_gap.parquet',
    # Old meta files (schema changed: now includes FFW fields)
    'meta.parquet',
    'meta_proforma.parquet',
    'tse_universe_meta.parquet',
    # Old intermediate files
    '_avgclose_primary_ref.parquet',
    '_avgclose_robust_ref.parquet',
    '_meta_endp_snapshot.parquet',
    '_meta_endr_snapshot.parquet',
    '_screen_candidates_1m.parquet',
    '_screen_topix_1m.parquet',
    'quarantine_low_coverage.parquet',
    'quarantine_low_coverage_robust.parquet',
    'attrition_original_today.parquet',
]

found = [f for f in OLD_FILES if os.path.exists(p(f))]
if not found:
    print('Cleanup: no target files present.')
elif DRY_RUN:
    print(f'Cleanup DRY_RUN — would delete {len(found)} file(s):')
    for f in found:
        print(f'  {f}')
    print('Set DRY_RUN=False above and re-run this cell to actually delete.')
else:
    for f in found:
        os.remove(p(f))
    print(f'Deleted {len(found)} file(s):')
    for f in found:
        print(f'  {f}')


### Step 1 · Load pre-reform TOPIX constituents from JPX official list
Source: `topix_prereform_codes.csv` — 2,182 stocks extracted from the JPX publication
*"Constituent Changes TOPIX New Index Series (effective 29 October 2021)"*.
4-digit TSE codes are converted to Eikon RICs by appending `.T`.

In [ ]:
# ── Load pre-reform TOPIX constituents from JPX PDF-derived CSV ────────────────
# Source: "Constituent Changes TOPIX New Index Series (effective 29 October 2021)"
# Published by Tokyo Stock Exchange on 7 October 2021
# Contains 2,182 stocks. Codes converted to Eikon RICs by appending ".T".

PROJECT_DIR = r'C:\Users\moolb\OneDrive\Dokumente\HSG\Bachelorarbeit\Python Analyses'
CODES_CSV   = os.path.join(PROJECT_DIR, 'topix_prereform_codes.csv')

if not already_saved('constituents_raw.parquet'):
    codes_df = pd.read_csv(CODES_CSV, dtype={'Code': str})
    codes_df['Code'] = codes_df['Code'].str.strip()
    codes_df['RIC']  = codes_df['Code'] + '.T'

    codes_df.to_parquet(p('constituents_raw.parquet'), index=False)
    print(f'  Saved {len(codes_df)} constituents from JPX CSV.')

df_const = pd.read_parquet(p('constituents_raw.parquet'))
rics = df_const['RIC'].tolist()
print(f'Working universe: {len(rics)} RICs  (expected: 2,182)')
print(f'  First 5: {rics[:5]}')
print(f'  Last 5 : {rics[-5:]}')

### Step 2 · Fundamental snapshot (FF market cap + sector)
Fields: `TR.CompanyMarketCap`, `TR.FreeFloatPct`, `TR.TRBCIndustryGroup`, `TR.GICSSector`, `TR.CommonName`, `TR.ISIN`

**FF Market Cap** = CompanyMarketCap × FreeFloatPct / 100

Batched in groups of 100 RICs (Eikon `get_data` limit).

In [ ]:
# ── Fix 1 + Fix 2: Fetch fundamentals with strategic-holdings fields ──────────
# Also fetch monthly-avg close price for the reference month to compute FF_MktCap

if not already_saved('meta.parquet'):
    print('Fetching fundamentals: Name, ISIN, MarketCap, FFW fields, Sector ...')
    meta_frames = []
    n_meta = len(rics)

    meta_fields = [
        'TR.CommonName',
        'TR.ISIN',
        'TR.CompanyMarketCap',
        'TR.FreeFloatPct',
        'TR.TRBCIndustryGroup',
        'TR.GICSSector',
    ] + FFW_FIELDS   # SharesOutstanding, TreasurySharesOutstanding, PctSharesHeldbyStratHolders

    META_FRIENDLY = ['Name', 'ISIN', 'CompanyMarketCap', 'FreeFloatPct',
                     'TRBC_Industry', 'GICS_Sector',
                     'SharesOutstanding', 'TreasuryShares', 'StrategicHoldersPct']

    for i in range(0, n_meta, META_BATCH):
        batch = rics[i : i + META_BATCH]
        try:
            df_batch, err = ek.get_data(
                batch,
                meta_fields,
                {'SDate': START_P}
            )
            if err:
                print(f'  Batch {i//META_BATCH+1} warning: {err}')
            meta_frames.append(df_batch)
        except Exception as e:
            print(f'  Batch {i//META_BATCH+1} error: {e}')

        if (i // META_BATCH + 1) % 5 == 0:
            print(f'  Meta batch {i//META_BATCH+1}/{(n_meta+META_BATCH-1)//META_BATCH}')
        time.sleep(SLEEP)

    df_meta = pd.concat(meta_frames, ignore_index=True)
    df_meta = rename_eikon_cols(df_meta, META_FRIENDLY)

    # Coerce numeric columns
    for col in ['CompanyMarketCap', 'FreeFloatPct', 'SharesOutstanding',
                'TreasuryShares', 'StrategicHoldersPct']:
        df_meta[col] = pd.to_numeric(df_meta[col], errors='coerce')

    # ── Fix 1: Compute Adjusted FFW with fallback ─────────────────────────────
    df_meta = apply_adjusted_ffw(df_meta)

    # ── Fix 2: Monthly avg close price for the reference month ────────────────
    ref_start, ref_end = ref_month_range(END_P)
    print(f'  Reference month for primary FF_MktCap: {ref_start} to {ref_end}')

    if not already_saved('_avgclose_primary_ref.parquet'):
        res_ref = fetch_ts(rics, ['CLOSE'], ref_start, ref_end, label='refmonth_P')
        avg_close = res_ref['CLOSE'].mean(axis=0)    # mean daily close per stock
        avg_close.name = 'AvgClose'
        avg_close.to_frame().to_parquet(p('_avgclose_primary_ref.parquet'))
        print(f'  Saved reference-month avg close for {avg_close.notna().sum()} stocks')

    avg_close_p = read_parquet_series(p('_avgclose_primary_ref.parquet'), 'AvgClose')

    df_meta = df_meta.merge(
        pd.DataFrame({'RIC': avg_close_p.index, 'AvgClose': avg_close_p.values}),
        on='RIC', how='left'
    )

    # ── Fix 2: FF_MktCap = SharesOutstanding x Adjusted_FFW x AvgClose ────────
    df_meta = compute_ff_mktcap(df_meta, price_col='AvgClose')

    df_meta.to_parquet(p('meta.parquet'), index=False)
    print(f'  Saved meta for {df_meta["FF_MktCap"].notna().sum()} / {len(df_meta)} stocks.')
    print(f'  Adjusted_FFW coverage : {df_meta["Adjusted_FFW"].notna().sum()}')
    print(f'  Fallback used         : {df_meta["ffm_fallback"].sum()}')
    print(f'  SharesOutstanding cov : {df_meta["SharesOutstanding"].notna().sum()}')

df_meta = pd.read_parquet(p('meta.parquet'))
print(f'Meta loaded: {len(df_meta)} rows')
print(f'  FF_MktCap available : {df_meta["FF_MktCap"].notna().sum()}')
print(f'  Adjusted_FFW fallback: {df_meta["ffm_fallback"].sum()} stocks')
df_meta.head(3)

### Step 3 · Daily CLOSE + VOLUME — primary window (Apr 2021 – Mar 2022)

In [ ]:
if not already_saved('prices_primary.parquet'):
    print(f'Fetching primary CLOSE + VOLUME  {START_P} -> {END_P} ...')
    res = fetch_ts(rics, ['CLOSE', 'VOLUME'], START_P, END_P, label='primary')
    res['CLOSE'].to_parquet(p('prices_primary.parquet'))
    res['VOLUME'].to_parquet(p('volume_primary.parquet'))
    print(f'  Saved prices  {res["CLOSE"].shape}')
    print(f'  Saved volume  {res["VOLUME"].shape}')

df_close  = pd.read_parquet(p('prices_primary.parquet'))
df_volume = pd.read_parquet(p('volume_primary.parquet'))
print(f'Primary prices : {df_close.shape}   volume : {df_volume.shape}')

### Step 4 · Build original (pre-reform) TOPIX with weights
Compute ATVR and FF-market-cap weights for the 2,182 pre-reform stocks.
This is the **"before"** baseline used in every analysis notebook.

In [ ]:
if not already_saved('constituents_original.parquet'):
    # De-duplicate meta (Eikon sometimes returns multiple rows per RIC)
    df_meta_dedup = df_meta.drop_duplicates(subset='RIC', keep='first')
    print(f'Meta de-duplicated: {len(df_meta)} -> {len(df_meta_dedup)} rows')

    # Merge meta onto constituents
    df = df_const.merge(df_meta_dedup, on='RIC', how='left')
    print(f'Merged constituents: {len(df)} rows (expected: {len(df_const)})')

    # ── Fix 4: Two-snapshot ATVR denominator ───────────────────────────────────
    # Denominator = average of FF_MktCap at START_P and END_P (without 0.75 factor)
    # We need meta snapshots at both window endpoints.
    # The main meta was pulled at SDate=START_P. We also need an END_P snapshot.

    SNAP_FRIENDLY = ['CompanyMarketCap', 'FreeFloatPct',
                     'SharesOutstanding', 'TreasuryShares', 'StrategicHoldersPct']

    if not already_saved('_meta_endp_snapshot.parquet'):
        print(f'Fetching END_P meta snapshot ({END_P}) for ATVR denominator ...')
        end_frames = []
        for i in range(0, len(rics), META_BATCH):
            batch = rics[i : i + META_BATCH]
            try:
                df_b, _ = ek.get_data(
                    batch,
                    ['TR.CompanyMarketCap', 'TR.FreeFloatPct'] + FFW_FIELDS,
                    {'SDate': END_P}
                )
                end_frames.append(df_b)
            except Exception:
                pass
            time.sleep(SLEEP)
        df_end = pd.concat(end_frames, ignore_index=True)
        df_end = rename_eikon_cols(df_end, SNAP_FRIENDLY)
        for col in df_end.columns[1:]:
            df_end[col] = pd.to_numeric(df_end[col], errors='coerce')
        df_end = apply_adjusted_ffw(df_end)
        # FF_MktCap at end = CompanyMarketCap * Adjusted_FFW (no 0.75, no monthly-avg price)
        df_end['FF_MktCap_end'] = df_end['CompanyMarketCap'] * df_end['Adjusted_FFW']
        df_end = df_end.drop_duplicates(subset='RIC', keep='first')
        df_end[['RIC', 'FF_MktCap_end']].to_parquet(p('_meta_endp_snapshot.parquet'), index=False)
        print(f'  Saved END_P snapshot for {df_end["FF_MktCap_end"].notna().sum()} stocks')

    df_end_snap = pd.read_parquet(p('_meta_endp_snapshot.parquet'))

    # FF_MktCap at start (from meta, using CompanyMarketCap * Adjusted_FFW, no monthly-avg)
    meta_idx = df_meta_dedup.set_index('RIC')
    ff_start = (meta_idx['CompanyMarketCap'] * meta_idx['Adjusted_FFW']).rename('FF_MktCap_start')

    end_idx = df_end_snap.set_index('RIC')['FF_MktCap_end']

    # Average of start and end
    avg_ff_denom = ((ff_start.reindex(meta_idx.index).fillna(0) +
                     end_idx.reindex(meta_idx.index).fillna(0)) / 2)
    # Where only one snapshot exists, use that one
    only_start = ff_start.notna() & end_idx.reindex(meta_idx.index).isna()
    only_end   = ff_start.isna() & end_idx.reindex(meta_idx.index).notna()
    avg_ff_denom[only_start] = ff_start[only_start]
    avg_ff_denom[only_end]   = end_idx.reindex(meta_idx.index)[only_end]
    avg_ff_denom[avg_ff_denom <= 0] = np.nan
    avg_ff_denom.name = 'FF_MktCap_denom'

    # ── Fix 5: Per-stock ATVR with coverage check ─────────────────────────────
    # Skip days where close OR volume is missing (don't fill)
    total_trading_days = df_close.shape[0]
    valid_mask = df_close.notna() & df_volume.notna()
    valid_days = valid_mask.sum(axis=0)            # per stock
    coverage   = valid_days / total_trading_days

    # Daily traded value only for valid days
    daily_tv = (df_close * df_volume).where(valid_mask)
    sum_tv   = daily_tv.sum(axis=0)

    # Per-stock annualisation scalar
    scalar = FULL_YEAR_DAYS / valid_days.replace(0, np.nan)
    ann_tv = sum_tv * scalar

    # ATVR = annualised traded value / avg FF_MktCap denominator
    atvr = (ann_tv / avg_ff_denom).reset_index()
    atvr.columns = ['RIC', 'ATVR']

    # Coverage info
    cov_df = pd.DataFrame({'RIC': valid_days.index, 'valid_days': valid_days.values,
                           'coverage': coverage.values})

    # Quarantine low-coverage stocks
    quarantine = cov_df[cov_df['coverage'] < MIN_COVERAGE].copy()
    if len(quarantine) > 0:
        quarantine.to_parquet(p('quarantine_low_coverage.parquet'), index=False)
        print(f'  Quarantined {len(quarantine)} stocks with coverage < {MIN_COVERAGE:.0%}')

    # Zero out ATVR for quarantined stocks
    quarantine_rics = set(quarantine['RIC'])
    atvr.loc[atvr['RIC'].isin(quarantine_rics), 'ATVR'] = np.nan

    df = df.merge(atvr, on='RIC', how='left')
    df = df.merge(cov_df, on='RIC', how='left')
    df['ATVR'] = pd.to_numeric(df['ATVR'], errors='coerce')

    # ── FF-market-cap weights (using screening FF_MktCap from Fix 2) ──────────
    df['w'] = df['FF_MktCap'] / df['FF_MktCap'].sum()

    df.to_parquet(p('constituents_original.parquet'), index=False)
    print(f'\nOriginal TOPIX saved: {len(df):,} stocks')
    print(f'  ATVR coverage : {df["ATVR"].notna().sum():,}')
    print(f'  FF_MktCap     : {df["FF_MktCap"].notna().sum():,}')
    print(f'  Median ATVR   : {df["ATVR"].median():.3f}')
    print(f'  Trading days  : {total_trading_days}')
    print(f'  Quarantined   : {len(quarantine):,} (coverage < {MIN_COVERAGE:.0%})')

df_orig = pd.read_parquet(p('constituents_original.parquet'))
print(f'Original TOPIX: {len(df_orig):,} stocks, '
      f'ATVR coverage: {df_orig["ATVR"].notna().sum():,}, '
      f'median ATVR: {df_orig["ATVR"].median():.3f}')

---
## Part B — Build Pro-forma Next-Gen TOPIX

### Step 5 · Load current TOPIX constituents
Source: `topix_current_codes.csv` — 1,674 stocks from JPX publication
*"Constituent Changes TOPIX New Index Series (effective 31 October 2025)"*.

In [ ]:
CURRENT_CSV = os.path.join(PROJECT_DIR, 'topix_current_codes.csv')

if not already_saved('constituents_current.parquet'):
    cur_df = pd.read_csv(CURRENT_CSV, dtype={'Code': str})
    cur_df['Code'] = cur_df['Code'].str.strip()
    cur_df['RIC']  = cur_df['Code'] + '.T'
    cur_df.to_parquet(p('constituents_current.parquet'), index=False)
    print(f'  Saved {len(cur_df)} current TOPIX codes.')

df_current = pd.read_parquet(p('constituents_current.parquet'))
rics_current = df_current['RIC'].tolist()
print(f'Current TOPIX: {len(rics_current)} stocks  (expected: 1,674)')

# Overlap with pre-reform universe
overlap     = set(rics) & set(rics_current)
new_since   = sorted(set(rics_current) - set(rics))
dropped     = len(set(rics) - set(rics_current))
print(f'  Overlap with pre-reform: {len(overlap)}')
print(f'  New since pre-reform   : {len(new_since)}')
print(f'  Dropped since reform   : {dropped}')
if new_since:
    print(f'  New RICs: {new_since[:10]}{"..." if len(new_since)>10 else ""}')

### Step 6 · TSE-wide FF market cap snapshot
Pull **only** `CompanyMarketCap` + `FreeFloatPct` for every plausible TSE code (1301–9999 + letter suffixes).
This is a cheap snapshot call — no time series. Used to:
1. Rank all TSE stocks by FF market cap
2. Identify non-TOPIX stocks large enough to be inclusion candidates

In [ ]:
# ── Fix 1 + Fix 2 + Fix 3: TSE-wide scan with adjusted FFW + two-stage pre-screening ──

if not already_saved('tse_universe_meta.parquet'):
    print('Scanning full TSE universe for FF market cap (with FFW fields) ...')

    # Generate all plausible TSE codes: 4-digit 1301–9999 + known letter suffixes
    all_codes = [str(c) for c in range(1301, 10000)]
    letter_suffixes = ['167A','250A','256A','262A','268A','285A','368A','409A','417A']
    all_codes.extend(letter_suffixes)
    all_rics = [c + '.T' for c in all_codes]

    frames = []
    n = len(all_rics)
    n_batches = (n + META_BATCH - 1) // META_BATCH

    tse_fields = ['TR.CompanyMarketCap', 'TR.FreeFloatPct'] + FFW_FIELDS
    TSE_FRIENDLY = ['CompanyMarketCap', 'FreeFloatPct',
                    'SharesOutstanding', 'TreasuryShares', 'StrategicHoldersPct']

    for i in range(0, n, META_BATCH):
        batch = all_rics[i : i + META_BATCH]
        try:
            df_b, _ = ek.get_data(
                batch,
                tse_fields,
                {'SDate': START_R}           # current-period snapshot
            )
            frames.append(df_b)
        except Exception:
            pass                              # many codes won't exist — that's fine

        batch_num = i // META_BATCH + 1
        if batch_num % 20 == 0 or batch_num == n_batches:
            print(f'  TSE scan  {batch_num}/{n_batches}  ({batch_num/n_batches*100:.0f}%)')
        time.sleep(SLEEP)

    df_tse = pd.concat(frames, ignore_index=True)
    df_tse = rename_eikon_cols(df_tse, TSE_FRIENDLY)

    for col in df_tse.columns[1:]:
        df_tse[col] = pd.to_numeric(df_tse[col], errors='coerce')

    # ── Fix 1: Adjusted FFW ───────────────────────────────────────────────────
    df_tse = apply_adjusted_ffw(df_tse)

    # ── Fix 2: FF_MktCap using monthly avg close for reference month ──────────
    ref_start_r, ref_end_r = ref_month_range(END_R)
    print(f'  Reference month for robustness FF_MktCap: {ref_start_r} to {ref_end_r}')

    # We'll fetch avg close for all valid TSE stocks (those with SharesOutstanding)
    valid_tse_rics = df_tse.dropna(subset=['CompanyMarketCap']).query('CompanyMarketCap > 0')['RIC'].tolist()

    if not already_saved('_avgclose_robust_ref.parquet'):
        print(f'  Fetching reference-month avg close for {len(valid_tse_rics)} TSE stocks ...')
        res_ref_r = fetch_ts(valid_tse_rics, ['CLOSE'], ref_start_r, ref_end_r, label='refmonth_R')
        avg_close_r = res_ref_r['CLOSE'].mean(axis=0)
        avg_close_r.name = 'AvgClose'
        avg_close_r.to_frame().to_parquet(p('_avgclose_robust_ref.parquet'))
        print(f'  Saved reference-month avg close for {avg_close_r.notna().sum()} stocks')

    avg_close_r = read_parquet_series(p('_avgclose_robust_ref.parquet'), 'AvgClose')

    df_tse = df_tse.merge(
        pd.DataFrame({'RIC': avg_close_r.index, 'AvgClose': avg_close_r.values}),
        on='RIC', how='left'
    )
    df_tse = compute_ff_mktcap(df_tse, price_col='AvgClose')

    # Keep only valid, positive FF market cap — these are real listed stocks
    df_tse = (df_tse.dropna(subset=['FF_MktCap'])
                    .query('FF_MktCap > 0')
                    .drop_duplicates(subset='RIC', keep='first')
                    .reset_index(drop=True))

    df_tse.to_parquet(p('tse_universe_meta.parquet'), index=False)
    print(f'  TSE universe: {len(df_tse):,} stocks with valid FF market cap')

df_tse = pd.read_parquet(p('tse_universe_meta.parquet'))
n_in_topix     = df_tse['RIC'].isin(rics_current).sum()
n_not_in_topix = len(df_tse) - n_in_topix
print(f'TSE universe: {len(df_tse):,} stocks')
print(f'  In current TOPIX : {n_in_topix:,}')
print(f'  Not in TOPIX     : {n_not_in_topix:,}')
if 'ffm_fallback' in df_tse.columns:
    print(f'  FFW fallback     : {df_tse["ffm_fallback"].sum():,}')

# ── Fix 3 Stage 1: Drop non-TOPIX stocks below 90th percentile of TSE-wide FF market cap
p90_ff = df_tse['FF_MktCap'].quantile(0.90)
print(f'\n90th percentile FF_MktCap (TSE-wide): {p90_ff/1e9:.2f}B')

non_topix = df_tse[~df_tse['RIC'].isin(rics_current)]
stage1_candidates = non_topix[non_topix['FF_MktCap'] >= p90_ff].copy()
print(f'Fix 3 Stage 1: Non-TOPIX stocks >= p90 → {len(stage1_candidates):,} candidates')
print(f'  (Dropped {len(non_topix) - len(stage1_candidates):,} below p90)')

# Save stage1 candidate RICs for use in the next cell (Stage 2)
rics_stage1_candidates = stage1_candidates['RIC'].tolist()
print(f'Stage 1 candidates ready for Stage 2 liquidity screening: {len(rics_stage1_candidates)}')

### Step 6b -- Fix 3 Stage 2: Liquidity pre-screening for inclusion candidates

In [ ]:
# ── Fix 3 Stage 2: Fetch 1 month of daily close+volume for Stage-1 candidates ─
# Drop those whose monthly traded value < 10th percentile of current TOPIX members'
# monthly traded value. Only survivors get full robustness window data.

ref_start_r, ref_end_r = ref_month_range(END_R)

# --- Fetch 1-month data for Stage 1 candidates ---
if not already_saved('_screen_candidates_1m.parquet'):
    print(f'Stage 2: Fetching 1-month CLOSE+VOLUME for {len(rics_stage1_candidates)} candidates ...')
    print(f'  Window: {ref_start_r} to {ref_end_r}')
    res_cand = fetch_ts(rics_stage1_candidates, ['CLOSE', 'VOLUME'],
                        ref_start_r, ref_end_r, label='cand_screen')
    # Monthly traded value per candidate
    if not res_cand['CLOSE'].empty and not res_cand['VOLUME'].empty:
        mtv_cand = (res_cand['CLOSE'] * res_cand['VOLUME']).sum(axis=0)
    else:
        mtv_cand = pd.Series(dtype=float)
    mtv_cand.name = 'MonthlyTV'
    mtv_cand.to_frame().to_parquet(p('_screen_candidates_1m.parquet'))
    print(f'  Saved monthly traded value for {mtv_cand.notna().sum()} candidates')

mtv_cand = read_parquet_series(p('_screen_candidates_1m.parquet'), 'MonthlyTV')

# --- Fetch 1-month data for current TOPIX members (for the p10 threshold) ---
if not already_saved('_screen_topix_1m.parquet'):
    print(f'Stage 2: Fetching 1-month CLOSE+VOLUME for {len(rics_current)} current TOPIX members ...')
    res_topix = fetch_ts(rics_current, ['CLOSE', 'VOLUME'],
                         ref_start_r, ref_end_r, label='topix_screen')
    if not res_topix['CLOSE'].empty and not res_topix['VOLUME'].empty:
        mtv_topix = (res_topix['CLOSE'] * res_topix['VOLUME']).sum(axis=0)
    else:
        mtv_topix = pd.Series(dtype=float)
    mtv_topix.name = 'MonthlyTV'
    mtv_topix.to_frame().to_parquet(p('_screen_topix_1m.parquet'))
    print(f'  Saved monthly traded value for {mtv_topix.notna().sum()} TOPIX members')

mtv_topix = read_parquet_series(p('_screen_topix_1m.parquet'), 'MonthlyTV')

# --- Apply the p10 liquidity threshold ---
p10_tv = mtv_topix.dropna().quantile(0.10)
print(f'\n10th percentile monthly traded value (current TOPIX): {p10_tv/1e6:.1f}M')

# Candidates surviving Stage 2
surviving_candidates = mtv_cand[mtv_cand >= p10_tv].index.tolist()
dropped_stage2 = len(rics_stage1_candidates) - len(surviving_candidates)
print(f'Stage 2 survivors: {len(surviving_candidates)} candidates')
print(f'  Dropped (below p10 traded value): {dropped_stage2}')

# Final candidate list: these get full robustness window data pull
rics_candidates = surviving_candidates
candidates = df_tse[df_tse['RIC'].isin(rics_candidates)].copy()

# Build the full list of stocks needing robustness-window time series
rics_robust_needed = sorted(set(rics_current + rics_candidates))
print(f'\nTotal stocks needing robustness data: {len(rics_robust_needed)}')

### Step 7 · Robustness window CLOSE + VOLUME (Apr 2025 – Mar 2026)
Pull daily data for all current TOPIX members + inclusion candidates.
Used to compute ATVR (for pro-forma building) and for all robustness-window analyses.

In [ ]:
# rics_robust_needed now contains only Stage-2 surviving candidates + current TOPIX
if not already_saved('prices_robust.parquet'):
    print(f'Fetching robustness CLOSE + VOLUME  {START_R} -> {END_R}')
    print(f'  Stocks to fetch: {len(rics_robust_needed)} (current TOPIX + screened candidates)')
    res_r = fetch_ts(rics_robust_needed, ['CLOSE', 'VOLUME'], START_R, END_R, label='robust')
    res_r['CLOSE'].to_parquet(p('prices_robust.parquet'))
    res_r['VOLUME'].to_parquet(p('volume_robust.parquet'))
    print(f'  Saved prices  {res_r["CLOSE"].shape}')
    print(f'  Saved volume  {res_r["VOLUME"].shape}')

df_close_r  = pd.read_parquet(p('prices_robust.parquet'))
df_volume_r = pd.read_parquet(p('volume_robust.parquet'))
print(f'Robustness prices: {df_close_r.shape}   volume: {df_volume_r.shape}')

### Step 8 · Build pro-forma Next-Gen TOPIX
Two-tier screening using robustness-window ATVR:
- **Continuation** (current TOPIX members): ATVR ≥ 0.14, top 97% cumulative FF market cap → **survivors**
- **Inclusion** (non-TOPIX TSE stocks): ATVR ≥ 0.20, top 96% cumulative FF market cap → **new additions**

Pro-forma = survivors + new additions, weighted by FF market cap.

In [ ]:
if not already_saved('constituents_proforma.parquet'):

    # ── Fix 4: Two-snapshot ATVR denominator for robustness window ─────────────
    # Need FF_MktCap at START_R and END_R (using Adjusted_FFW, no 0.75 factor)
    tse_idx = df_tse.drop_duplicates(subset='RIC').set_index('RIC')

    # START_R snapshot is already in df_tse (fetched at SDate=START_R)
    ff_start_r = (tse_idx['CompanyMarketCap'] * tse_idx['Adjusted_FFW']).rename('FF_MktCap_start')

    SNAP_FRIENDLY = ['CompanyMarketCap', 'FreeFloatPct',
                     'SharesOutstanding', 'TreasuryShares', 'StrategicHoldersPct']

    if not already_saved('_meta_endr_snapshot.parquet'):
        print(f'Fetching END_R meta snapshot ({END_R}) for ATVR denominator ...')
        end_frames_r = []
        all_robust_rics = rics_robust_needed
        for i in range(0, len(all_robust_rics), META_BATCH):
            batch = all_robust_rics[i : i + META_BATCH]
            try:
                df_b, _ = ek.get_data(
                    batch,
                    ['TR.CompanyMarketCap', 'TR.FreeFloatPct'] + FFW_FIELDS,
                    {'SDate': END_R}
                )
                end_frames_r.append(df_b)
            except Exception:
                pass
            time.sleep(SLEEP)
        df_end_r = pd.concat(end_frames_r, ignore_index=True)
        df_end_r = rename_eikon_cols(df_end_r, SNAP_FRIENDLY)
        for col in df_end_r.columns[1:]:
            df_end_r[col] = pd.to_numeric(df_end_r[col], errors='coerce')
        df_end_r = apply_adjusted_ffw(df_end_r)
        df_end_r['FF_MktCap_end'] = df_end_r['CompanyMarketCap'] * df_end_r['Adjusted_FFW']
        df_end_r = df_end_r.drop_duplicates(subset='RIC', keep='first')
        df_end_r[['RIC', 'FF_MktCap_end']].to_parquet(p('_meta_endr_snapshot.parquet'), index=False)
        print(f'  Saved END_R snapshot for {df_end_r["FF_MktCap_end"].notna().sum()} stocks')

    df_end_snap_r = pd.read_parquet(p('_meta_endr_snapshot.parquet'))
    end_idx_r = df_end_snap_r.set_index('RIC')['FF_MktCap_end']

    # Average of start and end for denominator
    all_rics_set = sorted(set(ff_start_r.index.tolist() + end_idx_r.index.tolist()))
    ff_s = ff_start_r.reindex(all_rics_set)
    ff_e = end_idx_r.reindex(all_rics_set)

    avg_ff_denom_r = pd.Series(index=all_rics_set, dtype=float)
    both = ff_s.notna() & ff_e.notna()
    avg_ff_denom_r[both] = (ff_s[both] + ff_e[both]) / 2
    only_s = ff_s.notna() & ff_e.isna()
    only_e = ff_s.isna() & ff_e.notna()
    avg_ff_denom_r[only_s] = ff_s[only_s]
    avg_ff_denom_r[only_e] = ff_e[only_e]
    avg_ff_denom_r[avg_ff_denom_r <= 0] = np.nan
    avg_ff_denom_r.name = 'FF_MktCap_denom'

    # ── Fix 5: Per-stock ATVR with coverage check (robustness) ────────────────
    total_trading_days_r = df_close_r.shape[0]
    valid_mask_r = df_close_r.notna() & df_volume_r.notna()
    valid_days_r = valid_mask_r.sum(axis=0)
    coverage_r   = valid_days_r / total_trading_days_r

    daily_tv_r = (df_close_r * df_volume_r).where(valid_mask_r)
    sum_tv_r   = daily_tv_r.sum(axis=0)

    # Per-stock annualisation
    scalar_r = FULL_YEAR_DAYS / valid_days_r.replace(0, np.nan)
    ann_tv_r = sum_tv_r * scalar_r

    # ATVR = annualised traded value / avg FF_MktCap denominator
    atvr_r = (ann_tv_r / avg_ff_denom_r).dropna()
    atvr_r.name = 'ATVR'

    # Coverage check — quarantine low-coverage stocks
    cov_df_r = pd.DataFrame({'RIC': valid_days_r.index, 'valid_days': valid_days_r.values,
                              'coverage': coverage_r.values})
    quarantine_r = cov_df_r[cov_df_r['coverage'] < MIN_COVERAGE].copy()
    if len(quarantine_r) > 0:
        quarantine_r.to_parquet(p('quarantine_low_coverage_robust.parquet'), index=False)
        print(f'  Quarantined {len(quarantine_r)} stocks with coverage < {MIN_COVERAGE:.0%}')

    quarantine_rics_r = set(quarantine_r['RIC'])
    atvr_r[atvr_r.index.isin(quarantine_rics_r)] = np.nan

    print(f'Robustness trading days: {total_trading_days_r}')
    print(f'ATVR computed for {atvr_r.notna().sum()} stocks')

    # ── Fix 2: Sequential procedure with SINGLE combined pool ─────────────────
    # Step A: ATVR gate per group (continuation vs inclusion)
    CONT_ATVR = 0.14
    INCL_ATVR = 0.20

    # -- Continuation candidates: current TOPIX members passing ATVR --
    df_cont = df_current.copy()
    df_cont['FF_MktCap'] = df_cont['RIC'].map(tse_idx['FF_MktCap'])
    df_cont['ATVR']      = df_cont['RIC'].map(atvr_r)
    df_cont['ATVR']      = pd.to_numeric(df_cont['ATVR'], errors='coerce')
    df_cont['coverage']  = df_cont['RIC'].map(cov_df_r.set_index('RIC')['coverage'])
    df_cont['stock_type'] = 'continuation'

    cont_atvr_pass = df_cont[
        (df_cont['ATVR'] >= CONT_ATVR) &
        (df_cont['FF_MktCap'].notna())
    ].copy()

    print(f'\n--- Step A: ATVR gate ---')
    print(f'  Continuation (ATVR >= {CONT_ATVR}): {len(cont_atvr_pass):,} / {len(df_cont):,}')

    # -- Inclusion candidates: non-TOPIX stocks passing ATVR --
    df_cand = candidates.copy()
    df_cand['ATVR'] = df_cand['RIC'].map(atvr_r)
    df_cand['ATVR'] = pd.to_numeric(df_cand['ATVR'], errors='coerce')
    df_cand['coverage'] = df_cand['RIC'].map(cov_df_r.set_index('RIC')['coverage'])
    df_cand['stock_type'] = 'inclusion'

    incl_atvr_pass = df_cand[
        (df_cand['ATVR'] >= INCL_ATVR) &
        (df_cand['FF_MktCap'].notna())
    ].copy()

    print(f'  Inclusion    (ATVR >= {INCL_ATVR}): {len(incl_atvr_pass):,} / {len(df_cand):,}')

    # ── Step B: Combine into ONE pool ──────────────────────────────────────────
    pool_cont = cont_atvr_pass[['RIC', 'FF_MktCap', 'ATVR', 'coverage', 'stock_type']].copy()
    pool_incl = incl_atvr_pass[['RIC', 'FF_MktCap', 'ATVR', 'coverage', 'stock_type']].copy()
    combined_pool = pd.concat([pool_cont, pool_incl], ignore_index=True)
    combined_pool = combined_pool.drop_duplicates(subset='RIC', keep='first')
    print(f'\n--- Step B: Combined pool: {len(combined_pool):,} stocks ---')

    # ── Step C: Cumulative FF market cap ranking on the combined pool ──────────
    combined_pool = combined_pool.sort_values('FF_MktCap', ascending=False).reset_index(drop=True)
    combined_pool['CumPct'] = (
        combined_pool['FF_MktCap'].cumsum() / combined_pool['FF_MktCap'].sum()
    )

    # ── Step D: Apply thresholds based on stock type ───────────────────────────
    # Continuation: top 97% cumulative (looser threshold)
    # Inclusion:    top 96% cumulative (stricter threshold)
    CONT_CAP = 0.97
    INCL_CAP = 0.96

    # Determine which stocks fall in top 96% and top 97% of the COMBINED pool
    idx_96 = int(combined_pool['CumPct'].searchsorted(INCL_CAP))
    idx_97 = int(combined_pool['CumPct'].searchsorted(CONT_CAP))
    rics_top96 = set(combined_pool.loc[:idx_96, 'RIC'])
    rics_top97 = set(combined_pool.loc[:idx_97, 'RIC'])
    print(f'  Combined pool: top 96% = {len(rics_top96):,}, top 97% = {len(rics_top97):,}')

    # Continuation stocks must be in top 97%
    survivors = combined_pool[
        (combined_pool['stock_type'] == 'continuation') &
        (combined_pool['RIC'].isin(rics_top97))
    ].copy()

    # Inclusion stocks must be in top 96%
    new_adds = combined_pool[
        (combined_pool['stock_type'] == 'inclusion') &
        (combined_pool['RIC'].isin(rics_top96))
    ].copy()

    print(f'\n--- Step D: Final selection ---')
    print(f'  Survivors (continuation, top {CONT_CAP:.0%}): {len(survivors):,}')
    print(f'  New additions (inclusion, top {INCL_CAP:.0%}) : {len(new_adds):,}')

    # ── Combine into pro-forma ─────────────────────────────────────────────────
    surv_df = survivors[['RIC', 'FF_MktCap', 'ATVR', 'coverage']].copy()
    surv_df['Code']   = surv_df['RIC'].str.replace('.T', '', regex=False)
    surv_df['source'] = 'continuation'

    if len(new_adds) > 0:
        adds_df = new_adds[['RIC', 'FF_MktCap', 'ATVR', 'coverage']].copy()
        adds_df['Code']   = adds_df['RIC'].str.replace('.T', '', regex=False)
        adds_df['source'] = 'inclusion'
        df_proforma = pd.concat([surv_df, adds_df], ignore_index=True)
    else:
        df_proforma = surv_df.copy()

    # FF-market-cap weights (using screening FF_MktCap from Fix 2)
    df_proforma['w'] = df_proforma['FF_MktCap'] / df_proforma['FF_MktCap'].sum()

    df_proforma.to_parquet(p('constituents_proforma.parquet'), index=False)

    print(f'\n{"="*55}')
    print(f'PRO-FORMA NEXT-GEN TOPIX')
    print(f'  Survivors (continuation) : {len(surv_df):,}')
    print(f'  New additions (inclusion): {len(new_adds):,}')
    print(f'  Total pro-forma          : {len(df_proforma):,}')
    print(f'  Median ATVR              : {df_proforma["ATVR"].median():.3f}')
    print(f'  Quarantined (low cov)    : {len(quarantine_r):,}')
    print(f'  Top-5 by weight:')
    top5 = df_proforma.nlargest(5, 'w')
    for _, row in top5.iterrows():
        print(f'    {row["RIC"]}  w={row["w"]:.4f}  FF={row["FF_MktCap"]/1e12:.2f}T')
    print(f'{"="*55}')

df_proforma = pd.read_parquet(p('constituents_proforma.parquet'))
rics_proforma = df_proforma['RIC'].tolist()
print(f'Pro-forma loaded: {len(df_proforma):,} stocks')
print(f'  Sources: {df_proforma["source"].value_counts().to_dict()}')

### Step 9 · Meta snapshot for pro-forma stocks (sector, name)
Pull `TR.CommonName`, `TR.ISIN`, `TR.TRBCIndustryGroup`, `TR.GICSSector` for all pro-forma stocks.
Uses the robustness-window date for the snapshot.

In [ ]:
# ── Fix 1: Fetch meta for pro-forma stocks with FFW fields ────────────────────
if not already_saved('meta_proforma.parquet'):
    print(f'Fetching meta for {len(rics_proforma)} pro-forma stocks ...')
    frames = []

    meta_pf_fields = [
        'TR.CommonName', 'TR.ISIN',
        'TR.CompanyMarketCap', 'TR.FreeFloatPct',
        'TR.TRBCIndustryGroup', 'TR.GICSSector',
    ] + FFW_FIELDS

    META_PF_FRIENDLY = ['Name', 'ISIN', 'CompanyMarketCap', 'FreeFloatPct',
                        'TRBC_Industry', 'GICS_Sector',
                        'SharesOutstanding', 'TreasuryShares', 'StrategicHoldersPct']

    for i in range(0, len(rics_proforma), META_BATCH):
        batch = rics_proforma[i : i + META_BATCH]
        try:
            df_b, _ = ek.get_data(
                batch,
                meta_pf_fields,
                {'SDate': START_R}
            )
            frames.append(df_b)
        except Exception as e:
            print(f'  Batch {i//META_BATCH+1} error: {e}')

        if (i // META_BATCH + 1) % 5 == 0:
            print(f'  Meta batch {i//META_BATCH+1}/{(len(rics_proforma)+META_BATCH-1)//META_BATCH}')
        time.sleep(SLEEP)

    df_meta_pf = pd.concat(frames, ignore_index=True)
    df_meta_pf = rename_eikon_cols(df_meta_pf, META_PF_FRIENDLY)

    for col in ['CompanyMarketCap', 'FreeFloatPct', 'SharesOutstanding',
                'TreasuryShares', 'StrategicHoldersPct']:
        df_meta_pf[col] = pd.to_numeric(df_meta_pf[col], errors='coerce')

    # Fix 1: Adjusted FFW
    df_meta_pf = apply_adjusted_ffw(df_meta_pf)

    # Fix 2: FF_MktCap using reference-month avg close
    # Reuse the robust reference-month avg close already saved
    if os.path.exists(p('_avgclose_robust_ref.parquet')):
        avg_close_pf = read_parquet_series(p('_avgclose_robust_ref.parquet'), 'AvgClose')
        df_meta_pf = df_meta_pf.merge(
            pd.DataFrame({'RIC': avg_close_pf.index, 'AvgClose': avg_close_pf.values}),
            on='RIC', how='left'
        )
        df_meta_pf = compute_ff_mktcap(df_meta_pf, price_col='AvgClose')
    else:
        # Fallback: use CompanyMarketCap * Adjusted_FFW
        df_meta_pf['FF_MktCap'] = df_meta_pf['CompanyMarketCap'] * df_meta_pf['Adjusted_FFW']

    df_meta_pf = df_meta_pf.drop_duplicates(subset='RIC', keep='first')

    df_meta_pf.to_parquet(p('meta_proforma.parquet'), index=False)
    print(f'  Saved meta for {len(df_meta_pf)} pro-forma stocks')
    print(f'  Sector coverage  : {df_meta_pf["TRBC_Industry"].notna().sum()}')
    print(f'  FFW fallback     : {df_meta_pf["ffm_fallback"].sum()}')

df_meta_pf = pd.read_parquet(p('meta_proforma.parquet'))
print(f'Pro-forma meta: {len(df_meta_pf)} stocks, '
      f'sector: {df_meta_pf["TRBC_Industry"].notna().sum()}')

# ── Enrich constituents_proforma with Name, sector (needed by analysis notebooks)
df_proforma = pd.read_parquet(p('constituents_proforma.parquet'))
merge_cols = ['RIC', 'Name', 'ISIN', 'TRBC_Industry', 'GICS_Sector']
available  = [c for c in merge_cols if c in df_meta_pf.columns]
df_proforma = df_proforma.drop(columns=[c for c in available if c in df_proforma.columns and c != 'RIC'],
                                errors='ignore')
df_proforma = df_proforma.merge(df_meta_pf[available], on='RIC', how='left')
df_proforma.to_parquet(p('constituents_proforma.parquet'), index=False)
print(f'Enriched pro-forma with Name + sector -> {len(df_proforma)} stocks')
print(f'  Name coverage   : {df_proforma["Name"].notna().sum()}')
print(f'  Sector coverage : {df_proforma["TRBC_Industry"].notna().sum()}')

### Step 10 · Daily BID + ASK — robustness window (for basket spread analysis)
Only fetched for pro-forma stocks (needed by notebook 06).

In [ ]:
def _has_useful_data(df):
    """True iff df has at least one non-NaN cell."""
    if df is None or df.empty:
        return False
    return bool(df.notna().to_numpy().any())


# Detect + purge all-NA bid/ask parquets from prior partial fetches so the
# snapshot fallback gets another chance. (Eikon can return a shaped frame of
# all NAs when a field is enabled but no trades were streamed in the window.)
for _f in ('bid_robust.parquet', 'ask_robust.parquet'):
    _fp = p(_f)
    if os.path.exists(_fp):
        _tmp = pd.read_parquet(_fp)
        if not _has_useful_data(_tmp):
            print(f'Detected all-NA {_f} — removing so snapshot fallback can run.')
            os.remove(_fp)

if not already_saved('bid_robust.parquet'):
    print(f'Fetching robustness BID + ASK for {len(rics_proforma)} pro-forma stocks ...')
    res_ba = fetch_ts(rics_proforma, ['BID', 'ASK'], START_R, END_R, label='bid/ask')

    bid_ts = res_ba.get('BID', pd.DataFrame())
    ask_ts = res_ba.get('ASK', pd.DataFrame())

    if not (_has_useful_data(bid_ts) and _has_useful_data(ask_ts)):
        print('  WARNING: BID/ASK time-series empty or all-NA.')
        print('  Falling back to snapshot via get_data ...')
        META_BATCH = 100
        ba_frames = []
        for i in range(0, len(rics_proforma), META_BATCH):
            batch = rics_proforma[i : i + META_BATCH]
            try:
                df_ba, _ = ek.get_data(batch, ['TR.BidPrice', 'TR.AskPrice'],
                                       {'SDate': END_R})
                ba_frames.append(df_ba)
            except Exception as e:
                print(f'    Batch {i//META_BATCH}: get_data failed: {e}')
            time.sleep(SLEEP)
        if ba_frames:
            df_ba = pd.concat(ba_frames, ignore_index=True)
            df_ba = rename_eikon_cols(df_ba, ['BID', 'ASK'])
            df_ba['BID'] = pd.to_numeric(df_ba['BID'], errors='coerce')
            df_ba['ASK'] = pd.to_numeric(df_ba['ASK'], errors='coerce')
            bid_snap = df_ba.set_index('RIC')['BID'].to_frame().T
            ask_snap = df_ba.set_index('RIC')['ASK'].to_frame().T
            bid_snap.index = pd.to_datetime([END_R])
            ask_snap.index = pd.to_datetime([END_R])
            if _has_useful_data(bid_snap) and _has_useful_data(ask_snap):
                bid_snap.to_parquet(p('bid_robust.parquet'))
                ask_snap.to_parquet(p('ask_robust.parquet'))
                print(f'  Saved snapshot bid/ask — coverage: '
                      f'{bid_snap.notna().sum().sum()} / {bid_snap.shape[1]}.')
            else:
                print('  FAILED: snapshot bid/ask also all NaN. '
                      '06_basket_spread.ipynb will skip S_basket analysis.')
        else:
            print('  FAILED: no snapshot batches succeeded.')
    else:
        bid_ts.to_parquet(p('bid_robust.parquet'))
        ask_ts.to_parquet(p('ask_robust.parquet'))
        print(f'  Saved bid {bid_ts.shape}  ask {ask_ts.shape}  '
              f'({bid_ts.notna().sum().sum()} non-NA bid cells)')

if os.path.exists(p('bid_robust.parquet')):
    df_bid = pd.read_parquet(p('bid_robust.parquet'))
    df_ask = pd.read_parquet(p('ask_robust.parquet'))
    print(f'Bid: {df_bid.shape}   Ask: {df_ask.shape}   '
          f'non-NA cells: bid={df_bid.notna().sum().sum():,}  '
          f'ask={df_ask.notna().sum().sum():,}')
else:
    print('No bid_robust.parquet on disk — 06_basket_spread.ipynb will skip analysis.')


### Step 11 · Gap CLOSE + VOLUME for pre-reform RICs (robustness window)

For the **original-today** counterfactual we need the 2,182 pre-reform members measured on the *same* 2025-26 window as the pro-forma index (so the only difference between the two indices is construction mechanics, not the measurement window).

`prices_robust.parquet` / `volume_robust.parquet` cover current TOPIX + Stage-2 inclusion candidates. For pre-reform RICs that were dropped in the reform (~600), we need an additional fetch here. Results go into a **gap** file that is concatenated with the main file at read-time via `load_robust_all()`.

Delisted RICs will naturally return empty series; they are filtered downstream in Steps 13-14 via the 85 % coverage rule.


In [ ]:
# ── Gap robust-window CLOSE+VOLUME fetch for pre-reform RICs ──────────────────
# Goal: every pre-reform RIC (2,182 total) should have a row in prices_robust_all
#       / volume_robust_all (= main + gap), so the original-today index can
#       measure ATVR/FF market cap on the robustness window exactly like the
#       pro-forma index does.

# Purge any zero-column gap parquets that slipped through from prior runs
# where Eikon returned an all-NaN frame (the dropna step collapses to 0 cols,
# which already_saved() then treats as "done"). Delete so we actually re-try.
for _f in ('prices_robust_gap.parquet', 'volume_robust_gap.parquet'):
    _fp = p(_f)
    if os.path.exists(_fp):
        _tmp = pd.read_parquet(_fp)
        if _tmp.shape[1] == 0:
            # Check whether there are pre-reform RICs still uncovered — if yes,
            # the empty gap is a failed fetch and we should retry.
            _raw = pd.read_parquet(p('constituents_raw.parquet'))
            _pre = set(_raw['RIC'])
            _main = set()
            if already_saved('prices_robust.parquet'):
                _main = set(pd.read_parquet(p('prices_robust.parquet')).columns)
            _gap_needed = len(_pre - _main)
            if _gap_needed > 0:
                print(f'Detected empty {_f} but {_gap_needed} pre-reform RICs '
                      f'still need coverage — removing so Step 11 re-runs.')
                os.remove(_fp)

if already_saved('prices_robust_gap.parquet') and already_saved('volume_robust_gap.parquet'):
    print('Step 11: gap files already present — skipping fetch.')
else:
    # Load existing robust prices (main file) to see what's already covered
    pr_existing = set()
    if already_saved('prices_robust.parquet'):
        pr_existing |= set(pd.read_parquet(p('prices_robust.parquet')).columns)

    # Pre-reform universe
    raw = pd.read_parquet(p('constituents_raw.parquet'))
    pre_rics = raw['RIC'].tolist()

    gap = sorted(set(pre_rics) - pr_existing)
    print(f'Pre-reform RICs           : {len(pre_rics)}')
    print(f'Already in prices_robust  : {len(pre_rics) - len(gap)}')
    print(f'Gap to fetch (robust)     : {len(gap)}')

    if not gap:
        # Save empty-but-typed placeholders so already_saved() returns True next time
        empty_df = pd.DataFrame(index=pd.date_range(START_R, END_R, freq='B'))
        empty_df.to_parquet(p('prices_robust_gap.parquet'))
        empty_df.to_parquet(p('volume_robust_gap.parquet'))
        print('Nothing to fetch; wrote empty gap files.')
    else:
        res = fetch_ts(gap, ['CLOSE', 'VOLUME'], START_R, END_R, label='pre_gap')
        prices_gap = res.get('CLOSE', pd.DataFrame()).dropna(axis=1, how='all')
        volume_gap = res.get('VOLUME', pd.DataFrame()).dropna(axis=1, how='all')

        # Guarantee at least an index range even when Eikon returned nothing
        if prices_gap.empty:
            prices_gap = pd.DataFrame(index=pd.date_range(START_R, END_R, freq='B'))
        if volume_gap.empty:
            volume_gap = pd.DataFrame(index=pd.date_range(START_R, END_R, freq='B'))

        prices_gap.to_parquet(p('prices_robust_gap.parquet'))
        volume_gap.to_parquet(p('volume_robust_gap.parquet'))
        print(f'Saved gap: prices {prices_gap.shape}, volume {volume_gap.shape}')
        if prices_gap.shape[1] == 0:
            print('  WARNING: gap fetch returned no data for any pre-reform-only RIC.')
            print('  The 471 pre-reform-only stocks will be quarantined as '
                  '"no_reference_price" in Step 14.')

# Smoke check
pr_all = load_robust_all('prices')
vol_all = load_robust_all('volume')
raw = pd.read_parquet(p('constituents_raw.parquet'))
pre_rics = set(raw['RIC'])
covered = pre_rics & set(pr_all.columns)
print(f'\nRobust prices now cover {len(covered)} / {len(pre_rics)} pre-reform RICs.')
print(f'  (missing {len(pre_rics) - len(covered)} — likely delisted since reform '
      f'or outside Eikon time-series coverage)')


### Step 11b · Gap BID + ASK for pre-reform RICs (robustness window)

Analogue of Step 11 for bid/ask. The Step 10 fetch covered only the main robust
universe (current TOPIX + Stage-2 inclusion candidates). Pre-reform RICs outside
that set need their own pull so the Basket Spread analysis can cover the full
pre-reform universe (same logic as for CLOSE+VOLUME).


In [ ]:
# ── Gap robust-window BID+ASK fetch via date-loop get_data ───────────────
# TICKET A-02: Step 10's time-series fetch returns empty for many pre-reform
# small-caps. Instead, loop over every trading day in [START_R, END_R] and pull
# TR.BidPrice / TR.AskPrice via get_data in batches of 100 RICs per date.
# Existing coverage (bid_robust_gap.parquet / ask_robust_gap.parquet) is
# preserved: only (date, RIC) cells still NaN are re-fetched.

# --- Gap universe: pre-reform RICs not covered by bid_robust.parquet --------
ba_main = set()
if already_saved('bid_robust.parquet'):
    ba_main |= set(pd.read_parquet(p('bid_robust.parquet')).columns)
raw = pd.read_parquet(p('constituents_raw.parquet'))
gap_rics = sorted(set(raw['RIC']) - ba_main)

# --- Trading-day axis --------------------------------------------------------
trading_days = pd.bdate_range(START_R, END_R)   # business-day calendar

# --- Load / seed the two matrices -------------------------------------------
def _seed(fname):
    if os.path.exists(p(fname)):
        df = pd.read_parquet(p(fname))
        df.index = pd.to_datetime(df.index)
    else:
        df = pd.DataFrame()
    # Expand to full (trading_days, gap_rics) grid so NaN cells == to-fetch
    df = df.reindex(index=df.index.union(trading_days),
                    columns=df.columns.union(gap_rics))
    df = df.sort_index()
    df = df.loc[trading_days, gap_rics]      # trim to target grid
    return df

bid_mat = _seed('bid_robust_gap.parquet')
ask_mat = _seed('ask_robust_gap.parquet')

# --- Decide whether there is anything to do ---------------------------------
# We skip wholesale only if BOTH matrices are fully populated for every gap RIC
# on every trading day. In practice that almost never happens on first run;
# partial files are common (e.g. prior run aborted mid-way).
missing_cells_bid = int(bid_mat.isna().to_numpy().sum())
missing_cells_ask = int(ask_mat.isna().to_numpy().sum())
print(f'Step 11b — gap RICs: {len(gap_rics)}   trading days: {len(trading_days)}')
print(f'Missing cells  bid: {missing_cells_bid:,}   ask: {missing_cells_ask:,}')

if missing_cells_bid == 0 and missing_cells_ask == 0:
    print('Step 11b: gap bid/ask matrices already fully populated — skipping fetch.')
else:
    BATCH_A02 = 100
    n_dates = len(trading_days)
    for d_idx, d in enumerate(trading_days):
        d_str = d.strftime('%Y-%m-%d')
        # RICs with NaN on this date (in either matrix)
        need_bid = bid_mat.columns[bid_mat.loc[d].isna()].tolist()
        need_ask = ask_mat.columns[ask_mat.loc[d].isna()].tolist()
        todo = sorted(set(need_bid) | set(need_ask))
        if not todo:
            continue

        for i in range(0, len(todo), BATCH_A02):
            chunk = todo[i : i + BATCH_A02]
            try:
                df, err = ek.get_data(
                    chunk,
                    ['TR.BidPrice', 'TR.AskPrice'],
                    {'SDate': d_str, 'EDate': d_str},
                )
                if df is None or df.empty:
                    continue
                df = rename_eikon_cols(df, ['BID', 'ASK'])
                df['BID'] = pd.to_numeric(df['BID'], errors='coerce')
                df['ASK'] = pd.to_numeric(df['ASK'], errors='coerce')
                # Fill the matrices
                for _, row in df.iterrows():
                    r = row['RIC']
                    if r in bid_mat.columns and pd.isna(bid_mat.at[d, r]):
                        bid_mat.at[d, r] = row['BID']
                    if r in ask_mat.columns and pd.isna(ask_mat.at[d, r]):
                        ask_mat.at[d, r] = row['ASK']
            except Exception as e:
                print(f'  {d_str} batch {i//BATCH_A02}: {type(e).__name__}: {e}')
            time.sleep(SLEEP)

        # Periodic checkpoint save (every 10 trading days)
        if (d_idx + 1) % 10 == 0 or d_idx == n_dates - 1:
            bid_mat.to_parquet(p('bid_robust_gap.parquet'))
            ask_mat.to_parquet(p('ask_robust_gap.parquet'))
            _bd = int(bid_mat.notna().to_numpy().sum())
            _ak = int(ask_mat.notna().to_numpy().sum())
            print(f'  [{d_idx+1:>3}/{n_dates}] saved   non-NA bid cells: {_bd:,}  ask: {_ak:,}')

    # Final save
    bid_mat.to_parquet(p('bid_robust_gap.parquet'))
    ask_mat.to_parquet(p('ask_robust_gap.parquet'))

# Smoke check — pre-reform coverage now
def _load_all_ba(which):
    frames = []
    for fn in (f'{which}_robust.parquet', f'{which}_robust_gap.parquet'):
        fp = p(fn)
        if os.path.exists(fp):
            frames.append(pd.read_parquet(fp))
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, axis=1)
    return df.loc[:, ~df.columns.duplicated()]

bid_all = _load_all_ba('bid')
pre_rics = set(pd.read_parquet(p('constituents_raw.parquet'))['RIC'])
have_any = {r for r in (pre_rics & set(bid_all.columns))
            if bid_all[r].notna().any()}
print(f'\nPre-reform RICs with ≥ 1 quoted bid: {len(have_any)} / {len(pre_rics)}')


### Step 12 · Gap END_R meta snapshot for pre-reform RICs

`meta_proforma.parquet` holds the END_R meta for current TOPIX; `tse_universe_meta.parquet` holds it for the TSE-wide scan. Together they cover 2,052 / 2,182 pre-reform members. The remaining ~130 RICs (almost all delisted) are attempted here; anything Eikon can't return is flagged as delisted in Step 13.


In [ ]:
# ── Gap END_R meta fetch for pre-reform RICs ──────────────────────────────────
if already_saved('meta_original_endr_gap.parquet'):
    print('Step 12: gap meta file already present — skipping fetch.')
else:
    # Already-covered RICs
    covered_endr = set()
    if already_saved('meta_proforma.parquet'):
        covered_endr |= set(pd.read_parquet(p('meta_proforma.parquet'))['RIC'])
    if already_saved('tse_universe_meta.parquet'):
        covered_endr |= set(pd.read_parquet(p('tse_universe_meta.parquet'))['RIC'])

    raw = pd.read_parquet(p('constituents_raw.parquet'))
    pre_rics = set(raw['RIC'])
    gap = sorted(pre_rics - covered_endr)
    print(f'Pre-reform RICs                     : {len(pre_rics)}')
    print(f'Already covered by existing meta    : {len(pre_rics) - len(gap)}')
    print(f'Gap to fetch (END_R meta snapshot)  : {len(gap)}')

    if not gap:
        empty_cols = ['RIC', 'Name', 'ISIN', 'CompanyMarketCap', 'FreeFloatPct',
                      'TRBC_Industry', 'GICS_Sector',
                      'SharesOutstanding', 'TreasuryShares', 'StrategicHoldersPct',
                      'AvgClose']
        pd.DataFrame(columns=empty_cols).to_parquet(p('meta_original_endr_gap.parquet'), index=False)
        print('Nothing to fetch; wrote empty gap file.')
    else:
        # --- Meta fields (same as Step 9 so the schemas concat cleanly) --------
        meta_fields = [
            'TR.CommonName',
            'TR.ISIN',
            'TR.CompanyMarketCap',
            'TR.FreeFloatPct',
            'TR.TRBCIndustryGroup',
            'TR.GICSSector',
        ] + FFW_FIELDS
        META_FRIENDLY = ['Name', 'ISIN', 'CompanyMarketCap', 'FreeFloatPct',
                         'TRBC_Industry', 'GICS_Sector',
                         'SharesOutstanding', 'TreasuryShares', 'StrategicHoldersPct']

        rows = []
        for i in range(0, len(gap), META_BATCH):
            batch = gap[i:i+META_BATCH]
            try:
                df_batch, err = ek.get_data(batch, meta_fields, {'SDate': END_R})
                if err:
                    print(f'  batch {i//META_BATCH+1} warning: {err}')
                if df_batch is not None and not df_batch.empty:
                    rows.append(df_batch)
            except Exception as exc:
                print(f'  batch {i//META_BATCH + 1}: {type(exc).__name__}: {exc}')
            time.sleep(SLEEP)

        if rows:
            meta_gap = pd.concat(rows, ignore_index=True)
            meta_gap = rename_eikon_cols(meta_gap, META_FRIENDLY)
            for col in ['CompanyMarketCap', 'FreeFloatPct', 'SharesOutstanding',
                        'TreasuryShares', 'StrategicHoldersPct']:
                if col in meta_gap.columns:
                    meta_gap[col] = pd.to_numeric(meta_gap[col], errors='coerce')
            meta_gap = apply_ffw(meta_gap, which='postreform')   # also computes post-reform FFW so schema matches tse_universe_meta

            # Reference-month AvgClose for FF mkt cap — piggyback on fetch_ts
            ref_start, ref_end = ref_month_range(END_R)
            try:
                res = fetch_ts(gap, ['CLOSE'], ref_start, ref_end, label='refmonth_gap')
                avg_close = res['CLOSE'].mean(axis=0) if not res['CLOSE'].empty else pd.Series(dtype=float)
                avg_df = pd.DataFrame({'RIC': avg_close.index, 'AvgClose': avg_close.values})
                meta_gap = meta_gap.merge(avg_df, on='RIC', how='left')
            except Exception as exc:
                print(f'  AvgClose fetch failed: {exc}')
                meta_gap['AvgClose'] = np.nan

            meta_gap = compute_ff_mktcap(meta_gap, price_col='AvgClose')
        else:
            empty_cols = ['RIC'] + META_FRIENDLY + ['Adjusted_FFW', 'ffm_fallback', 'AvgClose', 'FF_MktCap']
            meta_gap = pd.DataFrame(columns=empty_cols)

        meta_gap.to_parquet(p('meta_original_endr_gap.parquet'), index=False)
        print(f'Saved meta gap: {len(meta_gap)} rows for {len(gap)} requested RICs.')

# Smoke check
mg = pd.read_parquet(p('meta_original_endr_gap.parquet'))
print(f'\nmeta_original_endr_gap: {len(mg)} rows.')


### Step 13 · Build `meta_original_today.parquet`

Union of:
* `meta_proforma.parquet` — END_R meta for current TOPIX (1,674 RICs)
* `tse_universe_meta.parquet` — END_R meta for TSE-wide scan (~3,800 RICs)
* `meta_original_endr_gap.parquet` — gap fetched in Step 12

Deduped on RIC (preference order: `meta_proforma` → `tse_universe_meta` → gap), filtered to the 2,182 pre-reform RICs. Rows with no data at all are kept (RIC only, NaNs elsewhere) so the attrition waterfall in Step 15 can report them as "delisted".


In [ ]:
# ── Assemble meta_original_today (END_R meta for pre-reform universe) ────────
if already_saved('meta_original_today.parquet'):
    print('Step 13: meta_original_today.parquet already present — loading.')
else:
    raw = pd.read_parquet(p('constituents_raw.parquet'))
    pre_rics = raw['RIC'].tolist()

    # Preference order: meta_proforma > tse_universe_meta > gap
    meta_sources = []
    for fname in ['meta_proforma.parquet', 'tse_universe_meta.parquet', 'meta_original_endr_gap.parquet']:
        if already_saved(fname):
            df = pd.read_parquet(p(fname))
            df['_src'] = fname
            meta_sources.append(df)

    if not meta_sources:
        raise RuntimeError('No meta files available — run Steps 2, 6, 9, 12 first.')

    meta_all = pd.concat(meta_sources, ignore_index=True)
    # Keep first occurrence per RIC in the preference order above
    meta_all = meta_all.drop_duplicates(subset='RIC', keep='first')

    # Restrict to pre-reform universe + ensure every pre-reform RIC has a row
    meta_today = pd.DataFrame({'RIC': pre_rics}).merge(meta_all, on='RIC', how='left')

    # Apply PRE-reform FFW = FreeFloatPct / 100 (no deduction, no rounding)
    meta_today = apply_ffw(meta_today, which='prereform')

    meta_today.to_parquet(p('meta_original_today.parquet'), index=False)
    n_with_meta = meta_today['CompanyMarketCap'].notna().sum() if 'CompanyMarketCap' in meta_today.columns else 0
    print(f'Saved meta_original_today: {len(meta_today)} RICs total, {n_with_meta} with END_R meta.')

meta_today = pd.read_parquet(p('meta_original_today.parquet'))
print(f'\nmeta_original_today preview:')
print(meta_today.head(3))


### Step 14 · Build `constituents_original_today.parquet`

Inputs:
* `meta_original_today` (Step 13) — END_R meta + pre-reform FFW
* `prices_robust_all` / `volume_robust_all` (Step 3 + Step 11) — 2025-26 window

Pipeline (identical to pro-forma except for the FFW rule):
1. Compute FF market cap at END_R using pre-reform FFW.
2. Compute FF market cap at START_R via price × shares × FFW averaged across the reference month of April 2025 (two-snapshot denominator is the average of the two monthly-average FF market caps — same method as `02_calc_indexes.py` in JPX-style index design).
3. Per-stock ATVR on the robustness window with the 85 % coverage filter.
4. Quarantine reasons (mutually exclusive, checked in order):
   * `delisted` — no END_R meta at all
   * `missing_ffw` — FreeFloatPct is NaN
   * `low_coverage` — <85 % business-day coverage of daily CLOSE on the robust window
5. Weights = FF_MktCap / Σ(FF_MktCap) among survivors.

The output schema matches `constituents_proforma.parquet` so downstream analyses can treat the two indices symmetrically.


In [ ]:
# ── Build the original-today counterfactual index ─────────────────────────────
# Methodological note (2026-04): the pre-reform TOPIX had no investability
# screen, so a faithful no-reform counterfactual retains every pre-reform
# stock that is still listed in 2025/26 regardless of trading frequency.
# ATVR and coverage are computed and stored for transparency but are NOT
# used to quarantine. The only exclusions are:
#   • delisted          — no END_R meta available (stock gone)
#   • missing_ffw       — no FreeFloatPct available (can't compute weight)
#   • no_reference_price — zero trades in both April 2025 AND March 2026
#                          (two-snapshot denominator collapses to NaN)
if already_saved('constituents_original_today.parquet'):
    print('Step 14: constituents_original_today.parquet already present — loading.')
else:
    meta = pd.read_parquet(p('meta_original_today.parquet')).copy()

    # --- Quarantine flags (mutually exclusive, order matters) -----------------
    meta['quarantine_reason'] = ''
    mask_delisted = meta['CompanyMarketCap'].isna() if 'CompanyMarketCap' in meta.columns else pd.Series(True, index=meta.index)
    meta.loc[mask_delisted, 'quarantine_reason'] = 'delisted'

    mask_missing_ffw = meta['quarantine_reason'].eq('') & meta['FreeFloatPct'].isna()
    meta.loc[mask_missing_ffw, 'quarantine_reason'] = 'missing_ffw'

    # --- Load robust-window data ----------------------------------------------
    close = load_robust_all('prices')
    volume = load_robust_all('volume')

    # Restrict to pre-reform RICs that have any robust prices
    rics_here = [r for r in meta['RIC'] if r in close.columns]
    close_sub  = close[rics_here].copy()
    volume_sub = volume[rics_here].copy() if set(rics_here).issubset(volume.columns) else volume.reindex(columns=rics_here)

    # --- Two-snapshot FF market cap denominator -------------------------------
    start_dt = pd.Timestamp(START_R)
    first_month_start = start_dt.replace(day=1).strftime('%Y-%m-%d')
    first_month_end   = (start_dt.replace(day=1) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
    last_month_start, last_month_end = ref_month_range(END_R)

    avg_start = close_sub.loc[first_month_start:first_month_end].mean(axis=0)
    avg_end   = close_sub.loc[last_month_start:last_month_end].mean(axis=0)

    meta = meta.set_index('RIC')
    ff_start = avg_start * meta.reindex(avg_start.index)['SharesOutstanding'] * meta.reindex(avg_start.index)['Adjusted_FFW']
    ff_end   = avg_end   * meta.reindex(avg_end.index  )['SharesOutstanding'] * meta.reindex(avg_end.index  )['Adjusted_FFW']
    denom    = two_snapshot_denom(ff_start, ff_end)
    meta = meta.reset_index()

    # --- Diagnostic ATVR + coverage (NOT used as a filter) --------------------
    atvr, cov = compute_atvr(close_sub, volume_sub, denom,
                             min_cov=0.0,                # <<< accept all
                             trading_days=FULL_YEAR_DAYS)
    meta['ATVR']     = meta['RIC'].map(atvr.to_dict())
    meta['coverage'] = meta['RIC'].map(dict(zip(cov['RIC'], cov['coverage'])))

    # --- Flag no_reference_price (can't compute FF_MktCap) --------------------
    # A stock lacks a reference price if BOTH month-average closes are NaN.
    meta['AvgClose_Start'] = meta['RIC'].map(avg_start.to_dict())
    meta['AvgClose_End']   = meta['RIC'].map(avg_end.to_dict())
    mask_no_ref = (meta['quarantine_reason'].eq('')
                   & meta['AvgClose_Start'].isna()
                   & meta['AvgClose_End'].isna())
    meta.loc[mask_no_ref, 'quarantine_reason'] = 'no_reference_price'

    # --- FF market cap for weights -------------------------------------------
    # compute_ff_mktcap mutates `meta` in place; we drop any existing col first.
    if 'FF_MktCap' in meta.columns:
        meta = meta.drop(columns=['FF_MktCap'])
    compute_ff_mktcap(meta, price_col='AvgClose_End')

    # Blank out FF_MktCap for quarantined rows so downstream notebooks that
    # simply filter on FF_MktCap.notna() get only the investable set.
    survivors = meta['quarantine_reason'].eq('')
    meta.loc[~survivors, 'FF_MktCap'] = np.nan

    # --- Weights among survivors ---------------------------------------------
    total_ff  = meta.loc[survivors, 'FF_MktCap'].sum()
    meta['Weight'] = np.where(survivors, meta['FF_MktCap'] / total_ff, np.nan)

    # --- Column order (mirrors constituents_proforma.parquet) -----------------
    cols = ['RIC', 'Name', 'ISIN', 'GICS_Sector', 'TRBC_Industry',
            'SharesOutstanding', 'TreasuryShares', 'StrategicHoldersPct',
            'FreeFloatPct', 'Adjusted_FFW', 'ffm_fallback',
            'CompanyMarketCap', 'AvgClose_Start', 'AvgClose_End',
            'FF_MktCap', 'coverage', 'ATVR', 'Weight', 'quarantine_reason']
    present = [c for c in cols if c in meta.columns]
    meta = meta[present + [c for c in meta.columns if c not in present and not c.startswith('_')]]
    meta.to_parquet(p('constituents_original_today.parquet'), index=False)
    print(f'Saved constituents_original_today: {len(meta)} total, '
          f'{int(survivors.sum())} survivors, '
          f'Σweight={meta.loc[survivors, "Weight"].sum():.4f}')

cd_orig = pd.read_parquet(p('constituents_original_today.parquet'))
print(f'\nconstituents_original_today.parquet: {cd_orig.shape}')


### Step 15 · Attrition summary for the original-today index

Waterfall from the 2,182 pre-reform members to the final investable set used in the analysis notebooks. Each quarantine reason is mutually exclusive. Sanity check: Σ(Weight) among survivors should be 1.0 (float precision).


In [ ]:
# ── Attrition waterfall + top-10 weights sanity check ─────────────────────────
cd = pd.read_parquet(p('constituents_original_today.parquet'))
total = len(cd)
by_reason = cd['quarantine_reason'].value_counts(dropna=False).to_dict()
n_delisted   = by_reason.get('delisted', 0)
n_missing    = by_reason.get('missing_ffw', 0)
n_no_ref     = by_reason.get('no_reference_price', 0)
n_survivors  = by_reason.get('', 0)

print('Original-today attrition waterfall')
print('=' * 60)
print(f'  Pre-reform universe                : {total:>5}')
print(f'  − delisted (no END_R meta)         : {n_delisted:>5}')
print(f'  − missing FreeFloatPct             : {n_missing:>5}')
print(f'  − no reference price (both months) : {n_no_ref:>5}')
print(f'  = survivors (investable set)       : {n_survivors:>5}')
print(f'  Σ(Weight)                          : {cd["Weight"].sum():.6f}')
print()

# Diagnostic: how many survivors fall below the old 85% coverage threshold?
# These are retained (no investability filter) but flagged for transparency.
survivors = cd[cd['quarantine_reason'].eq('')]
n_below_85 = int((survivors['coverage'] < 0.85).sum())
n_below_50 = int((survivors['coverage'] < 0.50).sum())
print(f'Coverage diagnostics among survivors (INFORMATIONAL, not filters):')
print(f'  survivors with coverage <85%   : {n_below_85:>5} '
      f'({n_below_85/max(n_survivors,1)*100:.1f}%)')
print(f'  survivors with coverage <50%   : {n_below_50:>5} '
      f'({n_below_50/max(n_survivors,1)*100:.1f}%)')
print(f'  combined weight of <85% tail   : '
      f'{survivors.loc[survivors["coverage"] < 0.85, "Weight"].sum():.4%}')
print()

top10 = cd.nlargest(10, 'Weight')[['RIC', 'Name', 'GICS_Sector', 'Weight', 'FF_MktCap']]
print('Top-10 weights:')
print(top10.to_string(index=False))


In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
files = [
    # Core inputs
    ('constituents_raw.parquet',              'Pre-reform codes (2,182)',                   True),
    ('constituents_current.parquet',          'Current TOPIX codes (1,674)',                True),

    # Indices (output of this notebook, consumed by 01–06)
    ('constituents_original.parquet',         'Original TOPIX — primary window (context)',  True),
    ('constituents_original_today.parquet',   'Original-today counterfactual (analysis)',   True),
    ('constituents_proforma.parquet',         'Pro-forma Next-Gen TOPIX (analysis)',        True),

    # Meta snapshots
    ('meta.parquet',                          'Meta — pre-reform (2021 snapshot)',          True),
    ('meta_proforma.parquet',                 'Meta — current TOPIX @ END_R',               True),
    ('tse_universe_meta.parquet',             'Meta — TSE-wide scan @ END_R',               True),
    ('meta_original_today.parquet',           'Meta — pre-reform universe @ END_R',         True),
    ('meta_original_endr_gap.parquet',        'Meta — pre-reform gap fetch @ END_R',        False),

    # Daily time series — primary window (2021-04-01 .. 2022-03-31)
    ('prices_primary.parquet',                'CLOSE — primary window',                      True),
    ('volume_primary.parquet',                'VOLUME — primary window',                     True),

    # Daily time series — robustness window (2025-04-01 .. 2026-03-31)
    ('prices_robust.parquet',                 'CLOSE — robust wdw (current + candidates)',   True),
    ('volume_robust.parquet',                 'VOLUME — robust wdw (current + candidates)',  True),
    ('prices_robust_gap.parquet',             'CLOSE — robust wdw (pre-reform gap fill)',    False),
    ('volume_robust_gap.parquet',             'VOLUME — robust wdw (pre-reform gap fill)',   False),
    ('bid_robust.parquet',                    'BID — robust wdw',                            True),
    ('ask_robust.parquet',                    'ASK — robust wdw',                            True),
]

print('data/ folder contents:')
print(f'{"File":<45} {"Size":>8}  {"Req":>3}  Description')
print('-' * 100)
all_required_ok = True
for fname, desc, required in files:
    fp = p(fname)
    if os.path.exists(fp):
        size = f'{os.path.getsize(fp)/1e6:.1f} MB'
        marker = 'OK' if required else 'opt'
    else:
        size = 'MISSING'
        marker = 'OK' if not required else 'REQ'
        if required:
            all_required_ok = False
    req_flag = 'yes' if required else 'opt'
    print(f'  {fname:<43} {size:>8}  {req_flag:>3}  {desc}')

print()
if all_required_ok:
    print('All required files present. Ready to run analysis notebooks 01–06.')
else:
    print('WARNING: Some required files are missing — rerun the corresponding Steps.')
